### <font color = "gold"> UNA VEZ GENERADAS LAS COLUMNAS geocodigoFundar y geonombreFundar BORRAR LAS COLUMNAS DUPLICADAS </font>

In [1]:
import os
import pandas as pd

ruta_columnas_Geocodigo_descFundar_60percent =  '/home/capuccino/Desktop/TrabajoFundar/RENAMING/procesoRenaming/columnscsv_Geocodigo_descFunda_post_noteBookv2/24JUN25_columnscsv_Geocodigo_descFundar.csv' 

csvs_columnas_Geocodigo_descFundar_60percent = pd.read_csv(ruta_columnas_Geocodigo_descFundar_60percent)

csvs_columnas_Geocodigo_descFundar_60percent

,TOPICO,archivo_csv,columna_Geocodigo,columna_DescFundar
0,PESCAS,31_consumo_per_capita_pescado_anio_pais.csv,iso3,pais
1,PESCAS,16_valor_cantidad_precio_exportacion_pesquero.csv,NaN,NaN
2,PESCAS,01_produccion_pesquera_captura_y_acuicola_por_...,iso3,pais_nombre
3,PESCAS,22_pbg_pesquero.csv,geocodigo,provincia
4,PESCAS,04_desembarque_especie_ultimo_anio.csv,NaN,NaN
...,...,...,...,...
383,MERTRA,tasa_actividad_por_pais_anio.csv,iso3,iso3_desc
384,MERTRA,tasa_participacion_censos.csv,NaN,NaN
385,MERTRA,tiempo_social_trabajo_sexo_niveleducativo.csv,NaN,NaN
386,MERTRA,tasa_empleo_por_franja_etaria_anio_provincia.csv,NaN,prov_desc


In [2]:
import os
import pandas as pd

# 
SRC_BASE    = '/home/capuccino/Desktop/TrabajoFundar/RENAMING/procesoRenaming/new_data_argendata/data_SIN_BorradoColumnasDuplicadas'
DST_BASE    = '/home/capuccino/Desktop/TrabajoFundar/RENAMING/procesoRenaming/new_data_argendata/data_CON_BorradoColumnasDuplicadas'
MAPPING_CSV = ruta_columnas_Geocodigo_descFundar_60percent #las columnas de los csvs que cumple el 60% de coincidencia con el geocodico y desc_fundar, que
# son posteriormente usados para armar las columnas geocodigoFundar y geonombreFundar del respectivo csv

# creo carpeta topico destino principal
os.makedirs(DST_BASE, exist_ok = True)

# cargo mapeo de columnas a eliminar
df_map = pd.read_csv(MAPPING_CSV, encoding = 'utf-8', sep = ',')

# procesamos cada subtopico
for topic in os.listdir(SRC_BASE):
    topic_src = os.path.join(SRC_BASE, topic)
    if not os.path.isdir(topic_src): # ignoro todo lo que no es carpeta
        continue

    topic_dst = os.path.join(DST_BASE, topic)
    os.makedirs(topic_dst, exist_ok = True)

    # miro cada csv
    for fname in os.listdir(topic_src):
        if not fname.lower().endswith('.csv'):
            continue

        src_path = os.path.join(topic_src, fname)
        dst_path = os.path.join(topic_dst, fname)

        # se intenta abrir csvs con encoding y separador estandar
        try:
            df = pd.read_csv(src_path, encoding = 'utf-8', sep = ',')
        except Exception as e:
            print(f"Error leyendo {src_path}: {e}")
            # si falla, copiamos el archivo tal cual esta
            try:
                with open(src_path, 'rb') as fsrc, open(dst_path, 'wb') as fdst:
                    fdst.write(fsrc.read())
            except Exception as e2:
                print(f"Error copiando {src_path}: {e2}")
            continue

        # si existen las columnas geocodigoFundar y geonombreFundar, eliminar del csv las columnas de las cuales se armaron esas
        if 'geocodigoFundar' in df.columns and 'geonombreFundar' in df.columns:
            row = df_map[
                (df_map['TOPICO'] == topic) &
                (df_map['archivo_csv'] == fname)
            ]
            if not row.empty:
                cols_to_drop = []
                cod_col  = row.iloc[0]['columna_Geocodigo']
                desc_col = row.iloc[0]['columna_DescFundar']
                if pd.notna(cod_col) and cod_col in df.columns:
                    cols_to_drop.append(cod_col)
                if pd.notna(desc_col) and desc_col in df.columns:
                    cols_to_drop.append(desc_col)
                if cols_to_drop:
                    df = df.drop(columns = cols_to_drop)

        # reordeno columnas para que geocodigoFundar y geonombreFundar queden primero
        cols = df.columns.tolist()
        # recorres la lista fija ['geocodigoFundar', 'geonombreFundar']
        # si una de esas existe en cols, la añade a first_cols, en ese mismo orden
        first_cols = []
        for c in ['geocodigoFundar', 'geonombreFundar']:
            if c in cols:
                first_cols.append(c)
        
        resto_columnas = []
        for c in cols:
            # si una columna no está en first_cols, la guardo en resto_columnas
            if c not in first_cols:
                resto_columnas.append(c)       

        # aplico nuevo orden para que geocodigoFundar y geonombreFundar sean las primeras columnas de los datasets 
        df = df[first_cols + resto_columnas]           

        # guardamos csv en cada topico
        df.to_csv(dst_path, index = False, encoding = 'utf-8', sep = ',')


Error leyendo /home/capuccino/Desktop/TrabajoFundar/RENAMING/procesoRenaming/new_data_argendata/data_SIN_BorradoColumnasDuplicadas/MINERI/ranking_minerales_argentina.csv: 'utf-8' codec can't decode byte 0xe1 in position 17: invalid continuation byte


In [4]:
df_map

,TOPICO,archivo_csv,columna_Geocodigo,columna_DescFundar
0,PESCAS,31_consumo_per_capita_pescado_anio_pais.csv,iso3,pais
1,PESCAS,16_valor_cantidad_precio_exportacion_pesquero.csv,NaN,NaN
2,PESCAS,01_produccion_pesquera_captura_y_acuicola_por_...,iso3,pais_nombre
3,PESCAS,22_pbg_pesquero.csv,geocodigo,provincia
4,PESCAS,04_desembarque_especie_ultimo_anio.csv,NaN,NaN
...,...,...,...,...
383,MERTRA,tasa_actividad_por_pais_anio.csv,iso3,iso3_desc
384,MERTRA,tasa_participacion_censos.csv,NaN,NaN
385,MERTRA,tiempo_social_trabajo_sexo_niveleducativo.csv,NaN,NaN
386,MERTRA,tasa_empleo_por_franja_etaria_anio_provincia.csv,NaN,prov_desc
